# V22 - Phân tích khám phá dữ liệu

Notebook này trình bày phần EDA cho bộ dữ liệu đầu vào của pipeline V22. Trọng tâm là mô tả cấu trúc dữ liệu giao dịch, phân phối nhu cầu theo thời gian, mức độ thưa của từng SKU, tín hiệu trả hàng, lợi nhuận, giá bán/giá vốn và các biến ngoại sinh như thời tiết, CPI, WTI.

Các thống kê và biểu đồ trong notebook được chọn để giải thích vì sao pipeline V22 cần các bước preprocessing như gom giao dịch thành daily demand, dựng dense daily grid, tạo `days_since_first_sale`, tính `return_rate`, `profit_weight`, phân cụm SKU và bổ sung đặc trưng lịch/ngoại sinh.

Notebook chỉ phục vụ phân tích mô tả dữ liệu. Việc huấn luyện mô hình và sinh submission vẫn được chạy bằng terminal theo README.


## Cách đọc notebook

Các biểu đồ được chọn theo hướng EDA cho bài toán forecasting dạng retail/M5: mô tả chuỗi thời gian theo ngày, phân phối mức độ thưa của SKU, trả hàng, trọng số kinh doanh và các biến lịch/ngoại sinh. Các quan sát này nối trực tiếp với preprocessing trong V22 như dense daily grid, `days_since_first_sale`, `return_rate`, `profit_weight`, đặc trưng lịch Việt Nam và lag features.


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

if not (DATA_DIR / "train.csv").exists() and (PROJECT_DIR.parent / "data" / "train.csv").exists():
    DATA_DIR = PROJECT_DIR.parent / "data"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("train.csv exists:", (DATA_DIR / "train.csv").exists())


In [ ]:
def clean_numeric(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return series.astype(str).str.replace(",", ".", regex=False).astype(float)


def add_value_labels(ax, fmt="{:,.0f}"):
    for patch in ax.patches:
        height = patch.get_height()
        if np.isfinite(height):
            ax.annotate(
                fmt.format(height),
                (patch.get_x() + patch.get_width() / 2, height),
                ha="center",
                va="bottom",
                fontsize=9,
                xytext=(0, 3),
                textcoords="offset points",
            )


def file_status(name):
    path = DATA_DIR / name
    return {
        "file": name,
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else np.nan,
    }


## 1. Kiểm tra bộ dữ liệu đầu vào

Phần này cho thấy những file nào đang có trong `data/` và vai trò của chúng trong pipeline. Đây là bối cảnh tối thiểu để người đọc hiểu input trước khi xem preprocessing.


In [ ]:
expected_files = [
    "train.csv",
    "sample_submission.csv",
    "historical_weather.csv",
    "historical_cpi.csv",
    "wti_oil_prices_clean.csv",
    "sku_clustering_3_groups.csv",
]

files_df = pd.DataFrame([file_status(name) for name in expected_files])
display(files_df)


In [ ]:
train_path = DATA_DIR / "train.csv"
raw = pd.read_csv(train_path, low_memory=False)

print(f"Số dòng giao dịch: {len(raw):,}")
print(f"Số cột: {raw.shape[1]:,}")
display(raw.head())

schema = pd.DataFrame({
    "column": raw.columns,
    "dtype": [str(raw[c].dtype) for c in raw.columns],
    "missing": [raw[c].isna().sum() for c in raw.columns],
    "missing_pct": [raw[c].isna().mean() * 100 for c in raw.columns],
    "n_unique": [raw[c].nunique(dropna=True) for c in raw.columns],
})
display(schema)


## 2. Chuẩn hóa kiểu dữ liệu và tổng quan giao dịch

Pipeline V22 chuyển các cột số có thể dùng dấu phẩy thập phân sang `float`. EDA dưới đây áp dụng cùng nguyên tắc để các thống kê mô tả khớp với preprocessing.


In [ ]:
df = raw.copy()
numeric_cols = ["Quantity", "UnitPrice", "Unit Cost", "SalesAmount", "Cost Amount"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

df["Date"] = pd.to_datetime(df["Date"])
df["is_return"] = (df["Quantity"] < 0).astype(int)
df["positive_quantity"] = df["Quantity"].clip(lower=0)
df["return_quantity_abs"] = (-df["Quantity"].clip(upper=0))
df["Profit"] = df["SalesAmount"] - df["Cost Amount"]
df["margin_per_unit_proxy"] = df["UnitPrice"] - df["Unit Cost"]

summary = pd.DataFrame({
    "metric": [
        "date_min",
        "date_max",
        "transactions",
        "unique_skus",
        "positive_transactions",
        "return_transactions",
        "total_positive_quantity",
        "total_return_quantity_abs",
        "total_sales_amount",
        "total_cost_amount",
        "total_profit",
    ],
    "value": [
        df["Date"].min().date(),
        df["Date"].max().date(),
        f"{len(df):,}",
        f"{df['ItemCode'].nunique():,}",
        f"{(df['Quantity'] > 0).sum():,}",
        f"{df['is_return'].sum():,}",
        f"{df['positive_quantity'].sum():,.2f}",
        f"{df['return_quantity_abs'].sum():,.2f}",
        f"{df['SalesAmount'].sum():,.2f}",
        f"{df['Cost Amount'].sum():,.2f}",
        f"{df['Profit'].sum():,.2f}",
    ],
})
display(summary)


## 3. Cấu trúc file nộp bài

`sample_submission.csv` xác định đúng định dạng output mà các step trong V22 phải sinh ra. Việc kiểm tra file này giúp giải thích vì sao pipeline tách `validation` và `evaluation`, mỗi phần gồm 28 cột dự báo `F1..F28`.


In [ ]:
sample_path = DATA_DIR / "sample_submission.csv"
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    print("sample_submission shape:", sample.shape)
    display(sample.head())
    id_suffix = sample["id"].str.extract(r"_(validation|evaluation)$")[0].value_counts().rename_axis("split").reset_index(name="rows")
    display(id_suffix)
else:
    print("Không tìm thấy sample_submission.csv")


## 4. Xu hướng nhu cầu theo thời gian

Biểu đồ tổng lượng bán dương theo ngày giúp nhìn được seasonality, ngày cao điểm và các giai đoạn có nhu cầu thấp. Đây là lý do pipeline dùng đặc trưng lịch, ngày trong tuần, tháng, Tết, payday và các biến ngoại sinh.


In [ ]:
daily = df.groupby("Date").agg(
    positive_quantity=("positive_quantity", "sum"),
    return_quantity_abs=("return_quantity_abs", "sum"),
    sales_amount=("SalesAmount", "sum"),
    profit=("Profit", "sum"),
    active_skus=("ItemCode", "nunique"),
).reset_index()
daily["quantity_roll_28"] = daily["positive_quantity"].rolling(28, min_periods=1).mean()
daily["profit_roll_28"] = daily["profit"].rolling(28, min_periods=1).mean()

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
axes[0].plot(daily["Date"], daily["positive_quantity"], alpha=0.35, label="Daily quantity")
axes[0].plot(daily["Date"], daily["quantity_roll_28"], linewidth=2, label="28-day rolling mean")
axes[0].set_title("Total positive demand by day")
axes[0].set_ylabel("Quantity")
axes[0].legend()

axes[1].plot(daily["Date"], daily["active_skus"], color="#2f7ed8")
axes[1].set_title("Number of active SKUs by day")
axes[1].set_ylabel("Active SKUs")

axes[2].plot(daily["Date"], daily["profit"], alpha=0.35, label="Daily profit")
axes[2].plot(daily["Date"], daily["profit_roll_28"], linewidth=2, label="28-day rolling mean")
axes[2].set_title("Profit signal by day")
axes[2].set_ylabel("Profit")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
df["weekday"] = df["Date"].dt.day_name()
df["month"] = df["Date"].dt.to_period("M").astype(str)

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_stats = df[df["Quantity"] > 0].groupby("weekday")["Quantity"].sum().reindex(weekday_order)
monthly_stats = df[df["Quantity"] > 0].groupby("month")["Quantity"].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
weekday_stats.plot(kind="bar", ax=axes[0], color="#4c78a8")
axes[0].set_title("Positive quantity by weekday")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

monthly_stats.plot(ax=axes[1], color="#f58518")
axes[1].set_title("Positive quantity by month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Quantity")

plt.tight_layout()
plt.show()


## 5. Mức độ thưa và vòng đời SKU

Các SKU có rất nhiều ngày không bán là đặc điểm quan trọng của bài toán. V22 dựng dense daily grid để biểu diễn đầy đủ mọi ngày, đồng thời dùng `days_since_first_sale` để tránh xem các ngày trước khi sản phẩm xuất hiện là nhu cầu bằng 0.


In [ ]:
all_dates = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
total_days = len(all_dates)

positive = df[df["Quantity"] > 0].copy()
sku_stats = df.groupby("ItemCode").agg(
    transactions=("Quantity", "size"),
    positive_transactions=("Quantity", lambda s: (s > 0).sum()),
    return_transactions=("Quantity", lambda s: (s < 0).sum()),
    positive_quantity=("positive_quantity", "sum"),
    return_quantity_abs=("return_quantity_abs", "sum"),
    total_sales=("SalesAmount", "sum"),
    total_cost=("Cost Amount", "sum"),
    total_profit=("Profit", "sum"),
    avg_unit_price=("UnitPrice", "mean"),
    avg_unit_cost=("Unit Cost", "mean"),
).reset_index()

active_days = positive.groupby("ItemCode")["Date"].nunique().rename("active_days")
first_sale = positive.groupby("ItemCode")["Date"].min().rename("first_sale_date")
last_sale = positive.groupby("ItemCode")["Date"].max().rename("last_sale_date")

sku_stats = sku_stats.merge(active_days, on="ItemCode", how="left")
sku_stats = sku_stats.merge(first_sale, on="ItemCode", how="left")
sku_stats = sku_stats.merge(last_sale, on="ItemCode", how="left")
sku_stats["active_days"] = sku_stats["active_days"].fillna(0)
sku_stats["sparsity"] = 1 - sku_stats["active_days"] / total_days
sku_stats["return_ratio_tx"] = sku_stats["return_transactions"] / sku_stats["transactions"]
sku_stats["return_rate_qty"] = sku_stats["return_quantity_abs"] / (sku_stats["positive_quantity"] + 1e-5)
sku_stats["profit_weight_proxy"] = sku_stats["total_profit"].clip(lower=0)
sku_stats["avg_margin_proxy"] = sku_stats["avg_unit_price"] - sku_stats["avg_unit_cost"]

display(sku_stats.head())
display(sku_stats[["active_days", "sparsity", "positive_quantity", "return_rate_qty", "total_profit"]].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sku_stats["active_days"].hist(ax=axes[0], bins=60, color="#4c78a8")
axes[0].set_title("Distribution of active sales days per SKU")
axes[0].set_xlabel("Active days")
axes[0].set_ylabel("Number of SKUs")

sku_stats["sparsity"].hist(ax=axes[1], bins=60, color="#72b7b2")
axes[1].set_title("SKU sparsity distribution")
axes[1].set_xlabel("Sparsity = 1 - active_days / total_days")

sku_stats["positive_quantity"].clip(upper=sku_stats["positive_quantity"].quantile(0.99)).hist(ax=axes[2], bins=60, color="#f58518")
axes[2].set_title("Positive quantity distribution, clipped at p99")
axes[2].set_xlabel("Positive quantity")

plt.tight_layout()
plt.show()


## 6. Trả hàng và tín hiệu hoàn trả

Giao dịch `Quantity < 0` được V22 tách thành tín hiệu trả hàng. Nhóm SKU có tỷ lệ trả hàng cao có thể cần mô hình hóa khác vì forecast doanh số dương nhưng thực tế tồn tại dòng điều chỉnh âm.


In [ ]:
return_summary = pd.DataFrame({
    "metric": [
        "SKUs with at least one return",
        "Return transactions",
        "Return transaction ratio",
        "Absolute returned quantity",
        "Returned quantity / positive quantity",
    ],
    "value": [
        f"{(sku_stats['return_transactions'] > 0).sum():,}",
        f"{df['is_return'].sum():,}",
        f"{df['is_return'].mean() * 100:.4f}%",
        f"{df['return_quantity_abs'].sum():,.2f}",
        f"{df['return_quantity_abs'].sum() / (df['positive_quantity'].sum() + 1e-5) * 100:.4f}%",
    ],
})
display(return_summary)

top_return_skus = sku_stats.sort_values(["return_rate_qty", "return_quantity_abs"], ascending=False).head(15)
display(top_return_skus[["ItemCode", "transactions", "positive_quantity", "return_quantity_abs", "return_rate_qty", "total_profit"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

sku_stats["return_rate_qty"].clip(upper=sku_stats["return_rate_qty"].quantile(0.99)).hist(ax=axes[0], bins=60, color="#e45756")
axes[0].set_title("Return rate by quantity, clipped at p99")
axes[0].set_xlabel("Returned quantity / positive quantity")

daily[["Date", "positive_quantity", "return_quantity_abs"]].set_index("Date").plot(ax=axes[1], alpha=0.75)
axes[1].set_title("Positive quantity vs returned quantity by day")
axes[1].set_ylabel("Quantity")

plt.tight_layout()
plt.show()


## 7. Lợi nhuận và trọng số kinh doanh

Local scorer trong V22 dùng trọng số dựa trên lợi nhuận dương của SKU. Vì vậy EDA cần cho thấy phân phối lợi nhuận có lệch hay không và một nhóm SKU nhỏ có chiếm tỷ trọng lớn trong metric hay không.


In [ ]:
profit_df = sku_stats.copy()
profit_df["positive_profit"] = profit_df["total_profit"].clip(lower=0)
total_positive_profit = profit_df["positive_profit"].sum()
profit_df["profit_weight"] = profit_df["positive_profit"] / (total_positive_profit if total_positive_profit > 0 else 1)
profit_df = profit_df.sort_values("profit_weight", ascending=False).reset_index(drop=True)
profit_df["cum_profit_weight"] = profit_df["profit_weight"].cumsum()
profit_df["sku_rank_pct"] = (np.arange(len(profit_df)) + 1) / len(profit_df)

display(profit_df.head(20)[["ItemCode", "positive_quantity", "total_profit", "profit_weight", "cum_profit_weight", "sparsity"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
profit_df["positive_profit"].clip(upper=profit_df["positive_profit"].quantile(0.99)).hist(ax=axes[0], bins=60, color="#54a24b")
axes[0].set_title("Positive profit distribution, clipped at p99")
axes[0].set_xlabel("Positive profit")

axes[1].plot(profit_df["sku_rank_pct"] * 100, profit_df["cum_profit_weight"] * 100)
axes[1].set_title("Cumulative profit weight by SKU rank")
axes[1].set_xlabel("Top SKU rank (%)")
axes[1].set_ylabel("Cumulative profit weight (%)")
axes[1].axhline(80, color="red", linestyle="--", linewidth=1)
axes[1].axvline(20, color="red", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()


## 8. Giá, chi phí và biên lợi nhuận

Các biến `UnitPrice`, `Unit Cost` và margin được V22 dùng để tạo đặc trưng giá/margin. Phần này kiểm tra phân phối giá và mối quan hệ sơ bộ giữa lượng bán, giá và lợi nhuận.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df["UnitPrice"].clip(upper=df["UnitPrice"].quantile(0.99)).hist(ax=axes[0], bins=60, color="#4c78a8")
axes[0].set_title("UnitPrice distribution, clipped at p99")
axes[0].set_xlabel("UnitPrice")

df["Unit Cost"].clip(upper=df["Unit Cost"].quantile(0.99)).hist(ax=axes[1], bins=60, color="#f58518")
axes[1].set_title("Unit Cost distribution, clipped at p99")
axes[1].set_xlabel("Unit Cost")

df["margin_per_unit_proxy"].clip(
    lower=df["margin_per_unit_proxy"].quantile(0.01),
    upper=df["margin_per_unit_proxy"].quantile(0.99),
).hist(ax=axes[2], bins=60, color="#54a24b")
axes[2].set_title("Unit margin proxy distribution, p01-p99")
axes[2].set_xlabel("UnitPrice - Unit Cost")

plt.tight_layout()
plt.show()


In [ ]:
scatter_df = sku_stats[
    ["ItemCode", "positive_quantity", "avg_unit_price", "avg_margin_proxy", "total_profit", "sparsity"]
].replace([np.inf, -np.inf], np.nan).dropna()

if len(scatter_df) > 5000:
    scatter_df = scatter_df.sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].scatter(
    scatter_df["avg_unit_price"].clip(upper=sku_stats["avg_unit_price"].quantile(0.99)),
    scatter_df["positive_quantity"].clip(upper=sku_stats["positive_quantity"].quantile(0.99)),
    alpha=0.25,
    s=12,
)
axes[0].set_title("SKU quantity vs average unit price")
axes[0].set_xlabel("Average unit price, clipped")
axes[0].set_ylabel("Positive quantity, clipped")

axes[1].scatter(
    scatter_df["sparsity"],
    scatter_df["total_profit"].clip(lower=sku_stats["total_profit"].quantile(0.01), upper=sku_stats["total_profit"].quantile(0.99)),
    alpha=0.25,
    s=12,
    color="#e45756",
)
axes[1].set_title("SKU profit vs sparsity")
axes[1].set_xlabel("Sparsity")
axes[1].set_ylabel("Total profit, p01-p99")

plt.tight_layout()
plt.show()


## 9. Dữ liệu ngoại sinh: thời tiết, CPI và WTI

V22 có nhánh feature tích hợp thời tiết, CPI và giá dầu WTI nếu các file này tồn tại. Notebook chỉ kiểm tra độ phủ và trực quan hóa xu hướng chính, không merge vào training.


In [ ]:
external_specs = {
    "historical_weather.csv": "Date",
    "historical_cpi.csv": "Month",
    "wti_oil_prices_clean.csv": "Date",
}

for filename, date_col in external_specs.items():
    path = DATA_DIR / filename
    print("\n", "=" * 80)
    print(filename)
    if not path.exists():
        print("Không tìm thấy file.")
        continue
    ext = pd.read_csv(path)
    print("shape:", ext.shape)
    display(ext.head())
    display(pd.DataFrame({
        "column": ext.columns,
        "dtype": [str(ext[c].dtype) for c in ext.columns],
        "missing": [ext[c].isna().sum() for c in ext.columns],
    }))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

weather_path = DATA_DIR / "historical_weather.csv"
if weather_path.exists():
    weather = pd.read_csv(weather_path)
    weather["Date"] = pd.to_datetime(weather["Date"])
    for col in [c for c in ["temp_max", "temp_min", "rainfall"] if c in weather.columns]:
        axes[0].plot(weather["Date"], weather[col], label=col, alpha=0.8)
    axes[0].set_title("Historical weather signals")
    axes[0].legend()
else:
    axes[0].set_title("historical_weather.csv not found")

cpi_path = DATA_DIR / "historical_cpi.csv"
if cpi_path.exists():
    cpi = pd.read_csv(cpi_path)
    cpi["Date"] = pd.to_datetime(cpi["Month"])
    for col in [c for c in ["cpi_index", "cpi_mom_growth"] if c in cpi.columns]:
        axes[1].plot(cpi["Date"], cpi[col], marker="o", label=col)
    axes[1].set_title("Monthly CPI signals")
    axes[1].legend()
else:
    axes[1].set_title("historical_cpi.csv not found")

oil_path = DATA_DIR / "wti_oil_prices_clean.csv"
if oil_path.exists():
    oil = pd.read_csv(oil_path)
    oil["Date"] = pd.to_datetime(oil["Date"])
    if "wti_price" in oil.columns:
        axes[2].plot(oil["Date"], oil["wti_price"], color="#4c78a8")
    axes[2].set_title("WTI oil price")
else:
    axes[2].set_title("wti_oil_prices_clean.csv not found")

plt.tight_layout()
plt.show()


## 10. Hồ sơ cụm SKU

Nếu `sku_clustering_3_groups.csv` đã được tạo, phần này mô tả lại cụm SKU theo đúng các biến mà V22 dùng: sparsity, lượng bán trung bình, lợi nhuận tuyệt đối, tỷ lệ trả hàng và cờ có trả hàng.


In [ ]:
cluster_path = DATA_DIR / "sku_clustering_3_groups.csv"
if cluster_path.exists():
    clusters = pd.read_csv(cluster_path)
    print("cluster file shape:", clusters.shape)
    display(clusters.head())

    cluster_profile = clusters.groupby("cluster").agg(
        sku_count=("ItemCode", "count"),
        sparsity_mean=("sparsity", "mean"),
        avg_quantity_mean=("avg_quantity", "mean"),
        abs_profit_mean=("abs_profit", "mean"),
        return_ratio_mean=("return_ratio", "mean"),
        has_returns_sum=("has_returns", "sum"),
        profit_weight_sum=("profit_weight", "sum"),
    ).reset_index()
    cluster_profile["sku_pct"] = cluster_profile["sku_count"] / cluster_profile["sku_count"].sum() * 100
    display(cluster_profile)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    cluster_profile.set_index("cluster")["sku_count"].plot(kind="bar", ax=axes[0], color="#4c78a8")
    axes[0].set_title("SKU count by cluster")
    axes[0].set_xlabel("Cluster")
    add_value_labels(axes[0])

    cluster_profile.set_index("cluster")["sparsity_mean"].plot(kind="bar", ax=axes[1], color="#72b7b2")
    axes[1].set_title("Mean sparsity by cluster")
    axes[1].set_xlabel("Cluster")

    cluster_profile.set_index("cluster")["return_ratio_mean"].plot(kind="bar", ax=axes[2], color="#e45756")
    axes[2].set_title("Mean return ratio by cluster")
    axes[2].set_xlabel("Cluster")

    plt.tight_layout()
    plt.show()
else:
    print("Chưa có sku_clustering_3_groups.csv. Có thể chạy step0 từ terminal theo README nếu cần xem cụm.")


## 11. Liên hệ EDA với preprocessing V22

| Quan sát từ EDA | Cách V22 xử lý trong code |
| :-- | :-- |
| Dữ liệu là giao dịch rời rạc theo ngày và SKU | Gom transaction thành daily demand rồi dựng dense grid ngày x SKU trong `build_dense_grid` |
| SKU có nhiều ngày không bán | Tính `sparsity`, `active_days` và dùng phân cụm SKU ở Step 0 |
| Có SKU xuất hiện muộn | Tạo `days_since_first_sale` và đặt ngày trước khi SKU xuất hiện thành `NaN` để tránh zero-bias |
| Có giao dịch trả hàng | Tạo `return_rate`, `return_ratio`, `has_returns` để mô hình nhận biết nhóm trả hàng |
| Lợi nhuận phân bố lệch | Tạo `profit_weight` và dùng trong local WRMSSE/sample weighting |
| Nhu cầu biến động theo lịch | Thêm day-of-week, month, Tết, payday, mega-sale trong `create_shared_features` |
| Giá, chi phí, CPI, thời tiết, WTI có thể ảnh hưởng nhu cầu | Merge các biến giá/margin và ngoại sinh nếu file phụ trợ tồn tại |
| Dự báo có 2 horizon 28 ngày | Tách lag >= 28 cho Public và lag >= 56 cho Private để tránh leakage |
